# AI Personal Knowledge Base — Quickstart

Run each cell in order. Expected outputs are shown in comments above the cell.

**Prerequisites:** Ollama running with `qwen2.5:7b` (or `llama3`) pulled.

In [ ]:
# Cell 1: Environment check
# Expected: OK for all imports, or a clear error message with fix instructions
import sys
import requests

# Check Ollama
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    if not models:
        print("Ollama is running but no models found. Pull one:")
        print("  ollama pull qwen2.5:7b")
    else:
        print(f"Ollama OK — models: {', '.join(models[:5])}")
except Exception:
    print("Ollama is not running. Start it with: ollama serve")
    print("Download from https://ollama.com if not installed.")

# Check imports
try:
    from knowledge_base.config import Config
    from knowledge_base.ingestion import load_document, chunk_documents, ingest_directory
    from knowledge_base.retrieval import search
    from knowledge_base.generation import generate
    from knowledge_base.tracker import Tracker
    print("All imports OK")
except ImportError as e:
    print(f"Missing package: {e}")
    print("Run: pip install -r requirements.txt")

In [ ]:
# Cell 2: Load documents
# Expected: "Loaded N documents from ./documents/"
from knowledge_base.ingestion import load_directory
from knowledge_base.config import Config

config = Config(debug=True)

# Place your PDFs, .txt, and .md files in ./documents/ first
docs = load_directory("./documents", config)
print(f"Loaded {len(docs)} documents from ./documents/")
if docs:
    for i, doc in enumerate(docs[:3]):
        src = doc.metadata.get("source", "unknown")
        preview = doc.page_content[:100].replace("\n", " ")
        print(f"  [{i+1}] {src}: {preview}...")
else:
    print("No documents found. Add some to ./documents/ and re-run.")

In [ ]:
# Cell 3: Chunk documents
# Expected: "Chunked N documents into M chunks. Avg N tokens/chunk"
from knowledge_base.ingestion import chunk_documents

if docs:
    chunks = chunk_documents(docs, config)
    print(f"Chunked {len(docs)} documents into {len(chunks)} chunks")
    if chunks:
        avg_len = sum(len(c.page_content) for c in chunks) / len(chunks)
        print(f"Avg {avg_len:.0f} chars/chunk (limit: {config.chunk_size})")
else:
    print("Load documents first (Cell 2).")

In [ ]:
# Cell 4: Embed and store in Chroma
# Expected: "Embedded M chunks in X seconds. Stored in Chroma (collection: knowledge_base)"
# Note: first run downloads the embedding model (~80MB)
import time
from knowledge_base.ingestion import ingest_directory

t0 = time.time()
chunk_count, skipped = ingest_directory("./documents", config)
elapsed = time.time() - t0
print(f"Embedded {chunk_count} chunks in {elapsed:.1f}s")
print(f"Skipped {skipped} files. Stored in Chroma (collection: knowledge_base)")
print("Documents are now searchable.")

In [ ]:
# Cell 5: Retrieve relevant documents
# Expected: Top-4 result chunks with source metadata
from knowledge_base.retrieval import search

query = "Why is the sky blue?"
results = search(query, config)
print(f"Query: {query}")
print(f"Results: {len(results)}")
for i, doc in enumerate(results):
    src = doc.metadata.get("source", "unknown")
    preview = doc.page_content[:150].replace("\n", " ")
    print(f"\n  [{i+1}] {src}")
    print(f"      {preview}...")
if not results:
    print("No results. Try a different query or re-index your documents.")

In [ ]:
# Cell 6: Generate answer with citations
# Expected: Answer with numbered citations [1], [2], etc.
from knowledge_base.generation import generate

answer = generate(query, results, config)
print(f"Q: {query}")
print(f"A: {answer}")
print()
if "No relevant documents" in answer:
    print("Note: LLM found no answer in retrieved context.")
elif "[1]" in answer or "[2]" in answer:
    print("Citations detected.")

In [ ]:
# Cell 7: Track learning state
# Expected: "Logged Q&A to learning tracker. Session: session_XXXX"
from knowledge_base.tracker import Tracker

tracker = Tracker(config)
session_id = tracker.start_session(topic="Atmospheric Physics")

# Log this Q&A
tracker.log_qa(
    question=query,
    answer=answer,
    topic="Atmospheric Physics",
    source=results[0].metadata.get("source", "") if results else None,
    session_id=session_id,
    comprehension_notes="Rayleigh scattering is wavelength-dependent — shorter λ scatters more."
)
print(f"Logged Q&A to learning tracker. Session: {session_id}")

In [ ]:
# Cell 8: View learning stats
# Expected: Statistics table with topics, mastery bars, question counts
from knowledge_base.__main__ import cmd_stats
import argparse

args = argparse.Namespace(format="table")
cmd_stats(config, args)

tracker.end_session(session_id)
tracker.close()